# 01 — Multi-head Latent Attention (MLA), from scratch

Companion notebook to `../01-deepseek-architecture-deep-dive.md`.

This notebook implements a simplified, conceptual version of DeepSeek's Multi-head Latent Attention
in pure numpy: instead of caching a full-size key and value vector per attention head per token
(standard multi-head attention), it compresses the keys/values for a token into a single, shared,
lower-dimensional **latent** vector, caches *that*, and reconstructs full per-head keys/values from
the compressed latent representation only when actually computing attention.

The goal is to make the memory-savings argument from Chapter 1 numeric rather than just narrative:
we count how many numbers a standard KV cache stores per token versus how many an MLA-style cache
stores per token, at growing sequence lengths, and show attention still runs correctly off the
reconstructed keys/values.

Fully offline: numpy only, no model downloads, no GPU, no API keys.

In [1]:
import numpy as np

np.random.seed(42)

d_model = 512      # transformer hidden dimension
num_heads = 8
head_dim = 64       # standard MHA caches num_heads x head_dim keys AND values, per token
latent_dim = 96      # MLA caches ONE shared latent vector per token instead

standard_elems_per_token = num_heads * head_dim * 2   # keys + values, full per-head
mla_elems_per_token = latent_dim                        # the compressed latent, that's it

print(f"d_model={d_model}, num_heads={num_heads}, head_dim={head_dim}, latent_dim={latent_dim}")
print(f"Standard MHA per-token KV cache size (K+V, all heads): {standard_elems_per_token} elements")
print(f"MLA per-token KV cache size (compressed latent only):   {mla_elems_per_token} elements")
print(f"Compression ratio: {standard_elems_per_token / mla_elems_per_token:.2f}x smaller per token, per layer")


d_model=512, num_heads=8, head_dim=64, latent_dim=96
Standard MHA per-token KV cache size (K+V, all heads): 1024 elements
MLA per-token KV cache size (compressed latent only):   96 elements
Compression ratio: 10.67x smaller per token, per layer


## 1. The down/up-projection weights

MLA learns a **down-projection** that compresses a token's hidden state into the shared latent
representation that actually gets cached, and two **up-projections** that reconstruct full per-head
keys and values from that cached latent at attention-computation time. In a real trained model these
weights are learned end-to-end; here they're random, fixed matrices, which is enough to demonstrate
the *mechanism and the memory arithmetic* — the point of this notebook is the cache-size argument,
not reproducing trained model quality.

In [2]:
# Down-projection: hidden state -> compressed latent (THIS is what gets cached, per token)
W_dkv = np.random.randn(d_model, latent_dim) * 0.02

# Up-projections: compressed latent -> full per-head keys / values, reconstructed at attention time
W_uk = np.random.randn(latent_dim, num_heads * head_dim) * 0.02
W_uv = np.random.randn(latent_dim, num_heads * head_dim) * 0.02


def mla_encode(hidden_states: np.ndarray) -> np.ndarray:
    """hidden_states: (seq_len, d_model) -> compressed latent cache: (seq_len, latent_dim).
    This compressed representation is exactly what MLA writes into the KV cache -- not a
    full-size key/value pair per head."""
    return hidden_states @ W_dkv


def mla_reconstruct(latent_cache: np.ndarray):
    """Reconstruct full per-head K and V FROM the compressed latent cache, at attention time.
    The reconstruction cost is cheap relative to the memory saved by never caching full-size
    per-head keys/values for every past token."""
    seq_len = latent_cache.shape[0]
    K = (latent_cache @ W_uk).reshape(seq_len, num_heads, head_dim)
    V = (latent_cache @ W_uv).reshape(seq_len, num_heads, head_dim)
    return K, V


print("Weight shapes:")
print("  W_dkv (down-projection):", W_dkv.shape)
print("  W_uk   (up-projection, keys):", W_uk.shape)
print("  W_uv   (up-projection, values):", W_uv.shape)


Weight shapes:
  W_dkv (down-projection): (512, 96)
  W_uk   (up-projection, keys): (96, 512)
  W_uv   (up-projection, values): (96, 512)


## 2. Encode a sequence, cache the compressed latent, reconstruct, and run attention

A short synthetic sequence stands in for a chunk of tokens (e.g. an abstract being screened). We
compress it once into the latent cache, then show attention works correctly off keys/values
reconstructed from that compressed cache -- the compression isn't just smaller, it's still usable.

In [3]:
seq_len = 24
hidden_states = np.random.randn(seq_len, d_model) * 0.02

latent_cache = mla_encode(hidden_states)
print("Compressed KV cache shape (what MLA actually stores):", latent_cache.shape)
print("  -> total cached elements:", latent_cache.size)

K, V = mla_reconstruct(latent_cache)
print("Reconstructed per-head K shape:", K.shape, " V shape:", V.shape)

standard_cache_would_be = seq_len * standard_elems_per_token
print(f"\nA standard MHA cache for the same {seq_len}-token sequence would need "
      f"{standard_cache_would_be} elements; MLA's cache needs {latent_cache.size} -- "
      f"{standard_cache_would_be / latent_cache.size:.2f}x smaller.")


Compressed KV cache shape (what MLA actually stores): (24, 96)
  -> total cached elements: 2304
Reconstructed per-head K shape: (24, 8, 64)  V shape: (24, 8, 64)

A standard MHA cache for the same 24-token sequence would need 24576 elements; MLA's cache needs 2304 -- 10.67x smaller.


In [4]:
def attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> np.ndarray:
    """Standard causal scaled-dot-product attention, computed per head, off whatever K/V
    arrays it's given -- it doesn't know or care whether those K/V came from a full cache or
    were reconstructed from a compressed latent cache."""
    seq_len, n_heads, h_dim = Q.shape
    out = np.zeros_like(Q)
    scale = 1.0 / np.sqrt(h_dim)
    causal_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    for h in range(n_heads):
        scores = (Q[:, h, :] @ K[:, h, :].T) * scale
        scores = np.where(causal_mask, -1e9, scores)
        scores -= scores.max(axis=-1, keepdims=True)
        weights = np.exp(scores)
        weights /= weights.sum(axis=-1, keepdims=True)
        out[:, h, :] = weights @ V[:, h, :]
    return out


Q = np.random.randn(seq_len, num_heads, head_dim) * 0.02
attn_out = attention(Q, K, V)

print("Attention output shape:", attn_out.shape)
assert attn_out.shape == (seq_len, num_heads, head_dim)
assert np.isfinite(attn_out).all()
print("OK: attention ran successfully using keys/values reconstructed from the compressed "
      "latent cache -- the compression is usable, not just smaller.")


Attention output shape: (24, 8, 64)
OK: attention ran successfully using keys/values reconstructed from the compressed latent cache -- the compression is usable, not just smaller.


## 3. The memory argument, at scale

The savings above (at `seq_len=24`) are real but small in absolute terms. The argument that actually
matters for a production batch-screening pipeline (Chapter 4) is how this scales: the *per-token*
savings are constant, so the *total* cache-size gap grows linearly with sequence length and with how
many sequences are held concurrently in GPU memory -- exactly the situation a batch job screening many
abstracts in parallel creates.

In [5]:
print(f"{'seq_len':>8} | {'standard MHA cache (elements)':>30} | {'MLA cache (elements)':>22} | {'reduction':>10}")
print("-" * 80)
for sl in [128, 512, 2048, 8192, 32768]:
    standard = sl * standard_elems_per_token
    mla = sl * mla_elems_per_token
    print(f"{sl:>8} | {standard:>30,} | {mla:>22,} | {standard / mla:>9.2f}x")


 seq_len |  standard MHA cache (elements) |   MLA cache (elements) |  reduction
--------------------------------------------------------------------------------
     128 |                        131,072 |                 12,288 |     10.67x
     512 |                        524,288 |                 49,152 |     10.67x
    2048 |                      2,097,152 |                196,608 |     10.67x
    8192 |                      8,388,608 |                786,432 |     10.67x
   32768 |                     33,554,432 |              3,145,728 |     10.67x


## 4. Tying it back

- The **compressed latent cache** (Section 1-2) stores one shared, low-dimensional vector per token
  instead of a full-size key and value per attention head — a direct, mechanical reduction in what
  gets cached, not a vague "it's optimized" claim.
- **Attention still runs correctly** off keys/values reconstructed from that compressed cache
  (Section 2) — the compression is usable at inference time, which is the whole point: MLA isn't
  storing less and hoping it doesn't matter, it's storing a representation the model was trained to
  reconstruct from.
- The **memory-savings table** (Section 3) shows the gap is constant per token and therefore grows
  linearly with sequence length and concurrency — exactly the lever that matters for a batch
  screening pipeline trying to fit as many concurrent abstracts as possible into a fixed GPU memory
  budget (Chapter 4).